In [1]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")

if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

print("sys.path[0]:", sys.path[0])


sys.path[0]: C:\streaming_emulator


In [2]:
from pathlib import Path
import json

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
CONFIG_PATH = PIPELINE_ROOT / "replay" / "config" / "replay_config.json"

assert PIPELINE_ROOT.exists()
assert CONFIG_PATH.exists()

print("Pipeline root:", PIPELINE_ROOT)
print("Replay config:", CONFIG_PATH)


Pipeline root: C:\streaming_emulator
Replay config: C:\streaming_emulator\replay\config\replay_config.json


In [3]:
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    replay_config = json.load(f)

replay_config


{'enabled_sims': ['sim001',
  'sim002',
  'sim003',
  'sim007',
  'sim008',
  'sim009',
  'sim010'],
 'replay_mode': 'fixed_rate',
 'rows_per_second': 1,
 'batch_size': 100,
 'batch_interval_seconds': 5,
 'http_endpoint': 'http://127.0.0.1:8000/ingest/v1/events',
 'reset': {'enabled': False, 'archive_dlq': True}}

In [4]:
replay_config["enabled_sims"] = ["sim001", "sim002", "sim003", "sim007", "sim008", "sim009", "sim010"]
replay_config["replay_mode"] = "fixed_rate"
replay_config["rows_per_second"] = 1


replay_config["reset"] = {
    "enabled": False,
    "archive_dlq": True,
}

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(replay_config, f, indent=2)

print("Config updated.")


Config updated.


In [5]:
from replay.service.replay_service import ReplayService

service = ReplayService(
    pipeline_root=PIPELINE_ROOT,
    enabled_sims=replay_config.get("enabled_sims"),
    replay_mode=replay_config["replay_mode"],
    rows_per_second=replay_config.get("rows_per_second"),
    batch_interval_seconds=replay_config.get("batch_interval_seconds"),
    http_endpoint=replay_config["http_endpoint"],
)

service.start(
    reset=replay_config.get("reset", {}).get("enabled", False),
    archive_dlq=replay_config.get("reset", {}).get("archive_dlq", True),
)

print("Replay service started.")


Replay metrics server started on port 9001
Replay service started.


In [20]:
#service.wait()


In [7]:
service.stop_workers()
print("Replay workers stopped.")

Replay workers stopped.


In [24]:
service.reset(archive_dlq=True)
print("Replay state reset.")

Replay state reset.
